# OpERA field-processing agent
Tell it what to process in plain language → it finds the GoPro videos and GPX, checks their times line up, extracts frames,
writes GPS + echosounder depth into each JPG, then classifies the geotagged frames with the YOLO benthic model.

**Setup once**
1. Add your Gemini API key as a Colab secret named `GEMINI_API_KEY` (🔑 icon, left sidebar).
2. Upload survey data to Drive, e.g. `MyDrive/OpERA/surveys/13MAY2025/` containing the GoPro `.MP4` files and the echosounder `.GPX`.
3. Put the trained classification weights on Drive, e.g. `MyDrive/OpERA/models/yolo11l-benthic-cls.pt`
   (`best.pt` from `notebooks/YOLO_classification.ipynb`). Leave `YOLO_MODEL` empty to skip classification.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -q https://github.com/philspaceagency/sdmad-opera-fieldprocessing.git /content/opera 2>/dev/null || git -C /content/opera pull -q
!pip install -q -r /content/opera/requirements-yolo.txt

In [ ]:
import os, sys
from google.colab import userdata

DATA_ROOT  = '/content/drive/MyDrive/OpERA/surveys'
DOCS_INDEX = '/content/drive/MyDrive/OpERA/docs_index'                    # workflow-docs search index (built on first run)
YOLO_MODEL = '/content/drive/MyDrive/OpERA/models/yolo11l-benthic-cls.pt'  # '' → no classification

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
sys.path.insert(0, '/content/opera')

from opera_agent import OperaAgent
agent = OperaAgent(data_root=DATA_ROOT, docs_index=DOCS_INDEX,
                   model_path=YOLO_MODEL if YOLO_MODEL and os.path.exists(YOLO_MODEL) else None)

## Ask it to process

In [ ]:
print(agent.run("Check if the time range of the GPX matches the videos of the 13MAY2025 survey"))

In [ ]:
print(agent.run("Process the GoPro videos from the 13MAY2025 survey and give me the geotagged and classified images"))

Follow-ups keep context, for example:
- `"The camera clock was on UTC — redo it."` (the previous outputs in that folder are replaced)
- `"Only recording GH0123, one frame every 2 seconds, downscaled to 1920 px wide."`
- `"Reclassify with a confidence threshold of 0.5."`
- `"How did the original scripts name the frames?"` (answered from the repo)

In [ ]:
print(agent.run("Show me how many frames were geotagged per recording, the class counts, and where the untagged ones are."))

## Look at the result

In [ ]:
import pandas as pd
from IPython.display import Image, display
OUT = DATA_ROOT + '/13MAY2025_processed'   # the folder the agent reported
display(Image(f'{OUT}/qa_track_map.png'))
if os.path.exists(f'{OUT}/qa_class_map.png'):
    display(Image(f'{OUT}/qa_class_map.png'))
df = pd.read_csv(f'{OUT}/frame_data.csv')
print(df['benthic_class'].value_counts() if 'benthic_class' in df else df['status'].value_counts())
df.head()

## Without the LLM (same pipeline, direct call)
Useful for batch runs or when you already know the paths and settings.

In [ ]:
import json
from opera_agent import inspect, process_survey
from opera_agent.classify import classify_frames

VIDEOS = DATA_ROOT + '/13MAY2025'
print(json.dumps(inspect(VIDEOS), indent=1, default=str))       # check overlap first
# rep = process_survey(VIDEOS, VIDEOS + '_processed', interval_s=1, model_path=YOLO_MODEL or None)
# classify_frames(VIDEOS + '_processed', YOLO_MODEL, conf=0.5)   # re-classify only